In [ ]:
import numpy as np
import pandas as pd
from time import time
from tqdm import tqdm
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

from dirarte.datasets import FicoDataset, CreditDataset, GermanDataset, CompasDataset, BailDataset
from dirarte import FeatureTweakingExplainer, DirarteExplainer

In [2]:
def run(
    dataset, 
    estimator='XGBoost', 
    shift_type='split',
    cost_type='tlps',
    n_iter=10,
    max_samples=100,
):
    
    np.random.seed(0)
    results = {
        'dataset': [],
        'estimator': [],
        'shift_type': [],
        'cost_type': [],
        'n_samples': [],
        'method': [],
        'parameter': [],
        'prune_level': [],
        'merge_level': [],
        'validity (before)': [],
        'validity (after)': [],
        'cost': [],
        'plausibility': [],
        'sparsity': [],
        'time': [],     
        'time per samples': [],   
    }
    
    for i in tqdm(range(n_iter)):
        
        X_before, X_after, X_test, y_before, y_after, y_test = dataset.get_shifted_dataset(
            test_size=0.2, 
            shift_type=shift_type, 
            delete_fraction=0.1,
            label_shift_fraction=0.1
        )
        constraints = dataset.constraints

        if estimator == 'XGBoost':
            clf = XGBClassifier(n_estimators=100, max_depth=6).fit(X_before, y_before)
            clf_shift = XGBClassifier(n_estimators=100, max_depth=6).fit(X_after, y_after)
        else:
            clf = RandomForestClassifier(n_estimators=100, max_depth=6).fit(X_before, y_before)
            clf_shift = RandomForestClassifier(n_estimators=100, max_depth=6).fit(X_after, y_after)
        if max_samples > 0:
            X_target = X_test[clf.predict(X_test) == 0][:max_samples]
        else:
            X_target = X_test[clf.predict(X_test) == 0]
            
        for method in ['Baseline', 'DiRARTE-W', 'DiRARTE-X']:
            for prune_level in [1.0, 0.5]:
                for merge_level in [1, 2]:
                    
                    if method == 'Baseline':
                        epsilons = [0.0]
                        explainer = FeatureTweakingExplainer(
                            clf, 
                            constraints, 
                            cost_type=cost_type, 
                            prune_level=prune_level, 
                            merge_level=merge_level
                        )
                    
                    elif method == 'DiRARTE-W':
                        if estimator == 'XGBoost':
                            epsilons = [1.5, 2.0, 2.5, 3.0, 3.5]
                        else:
                            epsilons = [0.1, 0.2, 0.3, 0.4, 0.5]
                        explainer = DirarteExplainer(
                            clf, 
                            constraints, 
                            cost_type=cost_type, 
                            prune_level=prune_level, 
                            merge_level=merge_level, 
                            distribution='wasserstein', 
                            epsilon=epsilons
                        )
                        
                    else:
                        if estimator == 'XGBoost':
                            epsilons = [0.02, 0.04, 0.06, 0.08, 0.1]
                        else:
                            epsilons = [0.04, 0.08, 0.12, 0.16, 0.2]
                        explainer = DirarteExplainer(
                            clf, 
                            constraints, 
                            cost_type=cost_type, 
                            prune_level=prune_level, 
                            merge_level=merge_level, 
                            distribution='chi-squared', 
                            epsilon=epsilons
                        )

                    explainer = explainer.initialize(X_before)
                    time_dirarte = time()
                    recourses = explainer.explain_recourse(X_target)
                    time_dirarte = time() - time_dirarte
                    
                    if type(recourses) is not list:
                        recourses = [recourses]
                    
                    for epsilon, recourse in zip(epsilons, recourses):
                        X_cf = recourse.counterfactual
                        results['dataset'].append(dataset.name)
                        results['estimator'].append(estimator)
                        results['shift_type'].append(shift_type)
                        results['cost_type'].append(cost_type)
                        results['n_samples'].append(len(X_target))
                        results['method'].append(method)
                        results['parameter'].append(epsilon)
                        results['prune_level'].append(prune_level)
                        results['merge_level'].append(merge_level)
                        results['validity (before)'].append(clf.predict(X_cf).mean())
                        results['validity (after)'].append(clf_shift.predict(X_cf).mean())
                        results['cost'].append(recourse.get_average_cost())
                        results['plausibility'].append(recourse.get_average_plausibility())
                        results['sparsity'].append(recourse.get_average_sparsity())
                        results['time'].append(time_dirarte)
                        results['time per samples'].append(time_dirarte / len(X_target))
                
    return pd.DataFrame(results)

In [3]:
results_xg_tlps = []
for shift_type in ['split', 'delete', 'label']:
    for dataset in [FicoDataset(), CreditDataset(), GermanDataset(), CompasDataset(), BailDataset()]:
        result = run(
            dataset, 
            estimator='XGBoost', 
            shift_type=shift_type,
            cost_type='tlps',
        )
        results_xg_tlps.append(result)

100%|██████████| 10/10 [06:23<00:00, 38.39s/it]


In [4]:
results_rf_tlps = []
for shift_type in ['split', 'delete', 'label']:
    for dataset in [FicoDataset(), CreditDataset(), GermanDataset(), CompasDataset(), BailDataset()]:
        result = run(
            dataset, 
            estimator='RandomForest', 
            shift_type=shift_type,
            cost_type='tlps',
        )
        results_rf_tlps.append(result)

100%|██████████| 10/10 [11:51<00:00, 71.13s/it]


In [5]:
results_xg_std = []
for shift_type in ['split', 'delete', 'label']:
    for dataset in [FicoDataset(), CreditDataset(), GermanDataset(), CompasDataset(), BailDataset()]:
        result = run(
            dataset, 
            estimator='XGBoost', 
            shift_type=shift_type,
            cost_type='std',
        )
        results_xg_std.append(result)

100%|██████████| 10/10 [05:54<00:00, 35.40s/it]


In [6]:
results_rf_std = []
for shift_type in ['split', 'delete', 'label']:
    for dataset in [FicoDataset(), CreditDataset(), GermanDataset(), CompasDataset(), BailDataset()]:
        result = run(
            dataset, 
            estimator='RandomForest', 
            shift_type=shift_type,
            cost_type='std',
        )
        results_rf_std.append(result)

100%|██████████| 10/10 [11:04<00:00, 66.40s/it]


In [7]:
results = pd.concat(results_xg_tlps + results_rf_tlps + results_xg_std + results_rf_std, ignore_index=True)
results.to_csv('./results/results_ablation.csv', index=False)